# feature importance — Python demo

Numerical companion to the entry [feature importance](https://dictionaryofml.org/terms/featureimportance.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

Numerical companion to the glossary entry 'featureimportance'. The GeoSphere Austria weather station Krems (station id 3805, 48.42 N, 15.62 E) records six daily observations: maximum and minimum air temperature, mean air pressure, mean relative humidity, sunshine duration and precipitation. This script downloads the records for all 366 days of 2024 (dataset klima-v2-1d) and predicts the maximum daytime temperature of the next day from the twelve observations of the day before and of two days before: a data point is a day whose maximum temperature is the label, and its features are the twelve observations preceding it. Linear regression learns a linear hypothesis on the January--September data points; feature importance is evaluated on the distinct October--December scoring dataset. Permutation importance is then computed for each of the twelve features as the increase of the average squared error loss when the feature's values are randomly permuted across the data points. Both the calculation and its theoretical value agree that the day-before maximum temperature carries almost all of the credit. The day-before minimum temperature obtains a permutation importance of 0.00: the predictions do not depend on it. The same scores are then computed for a random forest (an ensemble of regression decision trees, grown here in numpy) learned on the same training data points: it reaches almost the same scoring loss, also ranks the day-before maximum temperature first, but spreads its reliance over several temperature features and uses the day-before minimum temperature, which the linear hypothesis ignores. Permutation importance needs the learned hypothesis only through its predictions on the scoring dataset, so it applies to both. A last block separates reliance from usefulness: the day-before minimum temperature correlates 0.69 with the label, as strongly as the two-days-before minimum, and still scores 0.00 under the linear hypothesis -- the score is a property of the hypothesis, not of the feature.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/featureimportance.py`](https://dictionaryofml.org/terms/featureimportance.py); CC BY 4.0.

Before running a downloaded copy, scan it: `uvx bandit -q -t B102,B307,B602,B605 featureimportance.py` (B102,B307,B602,B605 are the exec, eval and shell checks of [bandit](https://bandit.readthedocs.io/)) reports nothing: the script runs no shell command and evaluates no generated code.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "featureimportance.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""Feature importance for a next-day temperature forecast at Krems an der
Donau: which of the weather observations of the day before and of two
days before does
a learned linear hypothesis actually use?

Purpose
-------
Numerical companion to the glossary entry 'featureimportance'.  The
GeoSphere Austria weather station Krems (station id 3805, 48.42 N,
15.62 E) records six daily observations: maximum and minimum air
temperature, mean air pressure, mean relative humidity, sunshine
duration and precipitation.  This script downloads the records for all
366 days of 2024 (dataset klima-v2-1d) and predicts the maximum
daytime temperature of the next day from the twelve observations of
the day before and of two days before: a data point is a day whose
maximum temperature is the label, and its features are the twelve
observations preceding it.  Linear regression learns a linear
hypothesis on the January--September data points; feature importance is
evaluated on the distinct October--December scoring dataset.
Permutation importance is then computed for each of the twelve
features as the increase of the average squared error loss when the
feature's values are randomly permuted across the data points.  Both
the calculation and its theoretical value agree that the day-before maximum
temperature carries almost all of the credit. The day-before minimum temperature
obtains a permutation importance of 0.00: the
predictions do not depend on it.  The same scores are then computed for
a random forest (an ensemble of regression decision trees, grown here in
numpy) learned on the same training data points: it reaches almost the
same scoring loss, also ranks the day-before maximum temperature first, but
spreads its reliance over several temperature features and uses the day-before
minimum temperature, which the linear hypothesis ignores.  Permutation
importance needs the learned hypothesis only through its predictions on
the scoring dataset, so it applies to both.  A last block separates
reliance from usefulness: the day-before minimum temperature correlates 0.69
with the label, as strongly as the two-days-before minimum, and still
scores 0.00 under the linear hypothesis -- the score is a property of
the hypothesis, not of the feature.

Deterministic: the linear hypothesis is the closed-form solution of
linear regression; the permutations, the resampled copies and the
feature subsets of the random forest use fixed seeds.  Self-contained: numpy +
matplotlib only (stdlib urllib for the download).

Blocks
------
[B-fetch]   Download the six daily observations at Krems for 2024 from
            the GeoSphere data hub; write the 366 records to
            featureimportance_weather.csv and pin them to the archive.
[B-model]   Build 364 data points (features: the twelve observations of
            the day before and of two days before in their original units; label:
            the maximum daytime temperature of the next day); learn a
            linear hypothesis on January--September data points and reserve
            October--December as the scoring dataset; write the measured and
            predicted values for June 2024 to featureimportance_stem.csv.
[B-standard] Algorithm 1 of the entry: standardize the twelve columns of
            the training feature matrix by their sample mean and standard
            deviation, center the labels, learn by linear regression, and
            score each feature by its weight magnitude (the day-before maximum
            temperature 8.44, the two-days-before maximum 1.82).
[B-permute] Score each feature by its permutation importance (20
            permutation rounds, fixed seed); check that it matches
            twice the squared coefficient times the feature variance;
            write the scores to featureimportance_importance.csv.
[B-forest]  Learn a random forest (100 regression trees, each on a
            resampled copy of the training data points drawn with
            replacement, 4 features tried per
            split, depth at most 8, at least 5 data points per leaf) and
            score its twelve features by permutation importance on the
            same scoring dataset; write them in the order of the linear
            scores to featureimportance_rf.csv.
[B-useful]  Separate reliance from usefulness: the correlation of each
            feature with the label on the scoring dataset, a property of
            the feature and the data alone, set beside its two scores.
            The day-before minimum temperature correlates 0.69 with the label,
            as strongly as the two-days-before minimum, and still scores
            0.00 under the linear hypothesis (2.33 under the forest).
            Also the unit-free coefficient: the weight times the
            feature's standard deviation on the scoring dataset equals
            sqrt(I/2) for the linear hypothesis (4.96 = 4.96 for the day-before
            maximum temperature); write all columns to
            featureimportance_useful.csv.

Outputs
-------
featureimportance_weather.csv    : date + six observations, 366 days
featureimportance_stem.csv       : day, actual, predicted (June 2024)
featureimportance_importance.csv : idx, label, perm, theory (ascending
                                   by permutation importance, so the largest
                                   score sits rightmost in a stem plot;
                                   theory = 2 w_j^2 var(x_j) on the scoring
                                   dataset)
featureimportance_rf.csv         : idx, label, perm for the random forest,
                                   in the same idx order as the linear scores
featureimportance_useful.csv     : label, corr, linear, forest, wstd,
                                   sqrthalf, alg1 -- the correlation of each
                                   feature with the label on the scoring
                                   dataset, its two permutation scores, the
                                   weight times the scoring-set standard
                                   deviation, sqrt(linear/2), and the
                                   Algorithm 1 score on the training set
featureimportance.png            : preview (checking only) -- the June
                                   forecast and both sets of importance scores
"""

# /// script
# requires-python = ">=3.10"
# dependencies = ["matplotlib==3.10.7", "numpy==2.3.4"]
# ///

import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

report = []                         # collects (check name, pass/fail) pairs


def check(name, ok):                # records and prints one verification
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")

**[B-fetch]** Download the six daily observations at Krems for 2024 from the GeoSphere data hub; write the 366 records to featureimportance_weather.csv and pin them to the archive.

In [ ]:
PARAMS = ["tlmax", "tlmin", "p_mittel", "rf_mittel", "so_h", "rr"]
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       f"klima-v2-1d?parameters={','.join(PARAMS)}&station_ids=3805"
       "&start=2024-01-01&end=2024-12-31")
with urllib.request.urlopen(URL, timeout=120) as resp:
    payload = json.load(resp)
cols = payload["features"][0]["properties"]["parameters"]
days = [stamp[:10] for stamp in payload["timestamps"]]
raw = np.array([cols[p]["data"] for p in PARAMS], dtype=float).T
with open(OUT_DIR / "featureimportance_weather.csv", "w") as f:
    f.write("date," + ",".join(PARAMS) + "\n")
    for day, row in zip(days, raw):
        f.write(day + "," + ",".join(f"{v:g}" for v in row) + "\n")
check("[B-fetch] 366 days downloaded for 2024", len(days) == 366)
check("[B-fetch] no observation is missing", not np.isnan(raw).any())
check("[B-fetch] the record matches the archive (Jan 1: 9.6, 2.6, "
      "988.3, 67, 4.0, 0.0)",
      raw[0].tolist() == [9.6, 2.6, 988.3, 67.0, 4.0, 0.0])
raw[:, 5] = np.maximum(raw[:, 5], 0.0)   # the archive codes trace
# precipitation as -1.0 mm; a negative rainfall amount is a code, not a
# measurement, so it is set to zero

**[B-model]** Build 364 data points (features: the twelve observations of the day before and of two days before in their original units; label: the maximum daytime temperature of the next day); learn a linear hypothesis on January--September data points and reserve October--December as the scoring dataset; write the measured and predicted values for June 2024 to featureimportance_stem.csv.

In [ ]:
OBS = ["max temp", "min temp", "pressure", "humidity", "sunshine", "precip"]
labels = ([f"{o} (day before)" for o in OBS]
          + [f"{o} (two days before)" for o in OBS])
feats = np.array([np.concatenate([raw[t], raw[t - 1]])
                  for t in range(1, len(raw) - 1)])
y = raw[2:, 0]                       # tlmax of the following day
score_mask = np.array([day >= "2024-10-01" for day in days[2:]])
train_mask = ~score_mask
A = np.hstack([feats, np.ones((len(feats), 1))])
A_train, y_train = A[train_mask], y[train_mask]
A_score, y_score = A[score_mask], y[score_mask]
w, *_ = np.linalg.lstsq(A_train, y_train, rcond=None)
pred = A @ w
mse = float(np.mean((y_train - A_train @ w) ** 2))
var = float(np.mean((y_train - y_train.mean()) ** 2))
score_mse = float(np.mean((y_score - A_score @ w) ** 2))
print(f"  average squared error loss {mse:.2f}, label variance {var:.2f}")
print(f"  scoring dataset squared error loss {score_mse:.2f}")
check("[B-model] 364 data points with 12 features each",
      feats.shape == (364, 12))
check("[B-model] 272 training and 92 scoring data points",
      (int(train_mask.sum()), int(score_mask.sum())) == (272, 92))
check("[B-model] training loss 9.36 and label variance 77.12",
      (round(mse, 2), round(var, 2)) == (9.36, 77.12))
check("[B-model] scoring loss 8.16", round(score_mse, 2) == 8.16)

june = [i for i, t in enumerate(range(1, len(raw) - 1))
        if days[t + 1].startswith("2024-06")]
with open(OUT_DIR / "featureimportance_stem.csv", "w") as f:
    f.write("day,actual,predicted\n")
    for n, i in enumerate(june, start=1):
        f.write(f"{n},{y[i]:.1f},{pred[i]:.1f}\n")
check("[B-model] the stem plot covers the 30 days of June 2024",
      len(june) == 30)

**[B-standard]** Algorithm 1 of the entry: standardize the twelve columns of the training feature matrix by their sample mean and standard deviation, center the labels, learn by linear regression, and score each feature by its weight magnitude (the day-before maximum temperature 8.44, the two-days-before maximum 1.82).

In [ ]:
X_train = feats[train_mask]
Z_train = (X_train - X_train.mean(0)) / X_train.std(0)
w_std, *_ = np.linalg.lstsq(Z_train, y_train - y_train.mean(), rcond=None)
score_std = np.abs(w_std)
for i in np.argsort(score_std)[::-1][:4]:
    print(f"  Algorithm 1 score {labels[i]:22s} {score_std[i]:7.2f}")
check("[B-standard] the day-before max temp has the largest standardized weight "
      "magnitude, 8.44",
      int(np.argmax(score_std)) == labels.index("max temp (day before)")
      and round(float(score_std.max()), 2) == 8.44)
check("[B-standard] the two-days-before max temp is second with 1.82",
      round(float(score_std[labels.index("max temp (two days before)")]), 2) == 1.82
      and int(np.argsort(score_std)[-2]) == labels.index("max temp (two days before)"))
check("[B-standard] standardizing leaves the predictions unchanged: the "
      "weights equal the raw weights times the feature standard deviations",
      np.allclose(score_std, np.abs(w[:12]) * X_train.std(0)))

**[B-permute]** Score each feature by its permutation importance (20 permutation rounds, fixed seed); check that it matches twice the squared coefficient times the feature variance; write the scores to featureimportance_importance.csv.

In [ ]:
rng = np.random.default_rng(0)
perm = np.zeros(12)
first_perm = None                   # first permuted column, drawn in the entry
for j in range(12):
    inc = []
    for _ in range(20):
        Ap = A_score.copy()
        Ap[:, j] = rng.permutation(Ap[:, j])
        if first_perm is None:
            first_perm = Ap[:6, 0].copy()
        inc.append(float(np.mean((y_score - Ap @ w) ** 2)) - score_mse)
    perm[j] = np.mean(inc)
print("  first six rows of the scoring feature matrix, the day-before max temp:",
      A_score[:6, 0].tolist())
print("  the same entries after the first permutation of that column:",
      first_perm.tolist())
check("[B-permute] the scoring matrix starts with the day-before max temp "
      "15.4, 13.2, 13.1, 15.0, 12.1, 14.1",
      A_score[:6, 0].tolist() == [15.4, 13.2, 13.1, 15.0, 12.1, 14.1])
check("[B-permute] the first permutation moves 16.8, 18.1, 17.6, 4.0, "
      "14.1, 3.7 into those rows",
      first_perm.tolist() == [16.8, 18.1, 17.6, 4.0, 14.1, 3.7])
order = np.argsort(perm)             # ascending: largest ends up on the right
for i in order[::-1]:
    print(f"  perm. importance {labels[i]:22s} {perm[i]:7.2f}")
check("[B-permute] the largest permutation importance is the day-before max "
      "temp, 49.26", round(float(perm[int(order[-1])]), 2) == 49.26)
min_before = labels.index("min temp (day before)")
check("[B-permute] the day-before min temp has permutation importance 0.00",
      round(float(max(perm[min_before], 0.0)), 2) == 0.00)
theory = 2.0 * w[:12] ** 2 * feats[score_mask].var(0)
check("[B-permute] the three largest scores match twice the squared "
      "coefficient times feature variance within 0.5",
      all(abs(perm[i] - theory[i]) < 0.5 for i in order[-3:]))
with open(OUT_DIR / "featureimportance_importance.csv", "w") as f:
    f.write("idx,label,perm,theory\n")
    for n, i in enumerate(order):
        f.write(f"{n},{labels[i]},{perm[i]:.2f},{theory[i]:.2f}\n")

**[B-forest]** Learn a random forest (100 regression trees, each on a resampled copy of the training data points drawn with replacement, 4 features tried per split, depth at most 8, at least 5 data points per leaf) and score its twelve features by permutation importance on the same scoring dataset; write them in the order of the linear scores to featureimportance_rf.csv.

In [ ]:
X_score = feats[score_mask]


def grow_tree(X, y, rng, depth, min_leaf=5, n_try=4):
    # a regression tree: each split minimizes the summed squared error of
    # the two children over n_try randomly chosen features
    if depth == 0 or len(y) < 2 * min_leaf:
        return ("leaf", float(y.mean()))
    best = None
    for j in rng.choice(X.shape[1], n_try, replace=False):
        order_j = np.argsort(X[:, j])
        xs, ys = X[order_j, j], y[order_j]
        cs, cs2, n = np.cumsum(ys), np.cumsum(ys ** 2), len(ys)
        for k in range(min_leaf, n - min_leaf):
            if xs[k] == xs[k - 1]:
                continue
            left, right = cs[k - 1], cs[-1] - cs[k - 1]
            sse = ((cs2[k - 1] - left ** 2 / k)
                   + ((cs2[-1] - cs2[k - 1]) - right ** 2 / (n - k)))
            if best is None or sse < best[0]:
                best = (sse, int(j), (xs[k] + xs[k - 1]) / 2)
    if best is None:
        return ("leaf", float(y.mean()))
    _, j, thr = best
    left = X[:, j] <= thr
    return ("split", j, thr,
            grow_tree(X[left], y[left], rng, depth - 1, min_leaf, n_try),
            grow_tree(X[~left], y[~left], rng, depth - 1, min_leaf, n_try))


def tree_predict(tree, x):
    while tree[0] == "split":
        tree = tree[3] if x[tree[1]] <= tree[2] else tree[4]
    return tree[1]


rng = np.random.default_rng(0)
forest = []
for _ in range(100):
    boot = rng.integers(0, len(y_train), len(y_train))
    forest.append(grow_tree(X_train[boot], y_train[boot], rng, depth=8))


def forest_predict(X):
    return np.array([np.mean([tree_predict(t, x) for t in forest])
                     for x in X])


rf_mse = float(np.mean((y_score - forest_predict(X_score)) ** 2))
print(f"  random forest scoring dataset squared error loss {rf_mse:.2f}")
rng = np.random.default_rng(0)
perm_rf = np.zeros(12)
for j in range(12):
    inc = []
    for _ in range(20):
        Xp = X_score.copy()
        Xp[:, j] = rng.permutation(Xp[:, j])
        inc.append(float(np.mean((y_score - forest_predict(Xp)) ** 2))
                   - rf_mse)
    perm_rf[j] = np.mean(inc)
for i in np.argsort(perm_rf)[::-1]:
    print(f"  forest perm. importance {labels[i]:22s} {perm_rf[i]:7.2f}")
check("[B-forest] the forest's scoring loss 8.19 is within 0.1 of the "
      "linear hypothesis' 8.16", abs(rf_mse - score_mse) < 0.1
      and round(rf_mse, 2) == 8.19)
check("[B-forest] the day-before max temp ranks first for the forest too, "
      "with 14.93", int(np.argmax(perm_rf)) == labels.index("max temp (day before)")
      and round(float(perm_rf.max()), 2) == 14.93)
check("[B-forest] the forest uses the day-before min temp, score 2.33",
      round(float(perm_rf[min_before]), 2) == 2.33)
with open(OUT_DIR / "featureimportance_rf.csv", "w") as f:
    f.write("idx,label,perm\n")
    for n, i in enumerate(order):
        f.write(f"{n},{labels[i]},{perm_rf[i]:.2f}\n")

**[B-useful]** Separate reliance from usefulness: the correlation of each feature with the label on the scoring dataset, a property of the feature and the data alone, set beside its two scores. The day-before minimum temperature correlates 0.69 with the label, as strongly as the two-days-before minimum, and still scores 0.00 under the linear hypothesis (2.33 under the forest). Also the unit-free coefficient: the weight times the feature's standard deviation on the scoring dataset equals sqrt(I/2) for the linear hypothesis (4.96 = 4.96 for the day-before maximum temperature); write all columns to featureimportance_useful.csv.

In [ ]:
corr = np.array([np.corrcoef(X_score[:, j], y_score)[0, 1] for j in range(12)])
print(f"  {'feature':22s} {'corr(x,y)':>9s} {'linear I':>9s} {'forest I':>9s}")
for i in order[::-1]:
    print(f"  {labels[i]:22s} {corr[i]:9.2f} {perm[i]:9.2f} {perm_rf[i]:9.2f}")
min_before2 = labels.index("min temp (two days before)")
check("[B-useful] the day-before min temp correlates 0.69 with the label on "
      "the scoring dataset, as strongly as the two-days-before min temp",
      (round(float(corr[min_before]), 2), round(float(corr[min_before2]), 2))
      == (0.69, 0.69))
check("[B-useful] and still scores 0.00 (linear) and 2.33 (forest)",
      (round(float(max(perm[min_before], 0.0)), 2),
       round(float(perm_rf[min_before]), 2)) == (0.00, 2.33))
check("[B-useful] the day-before max temp has the largest correlation, 0.87",
      int(np.argmax(corr)) == labels.index("max temp (day before)")
      and round(float(corr.max()), 2) == 0.87)
wstd = np.abs(w[:12]) * X_score.std(0)      # prediction change per std
sqrthalf = np.sqrt(np.maximum(perm, 0.0) / 2.0)
max_before = labels.index("max temp (day before)")
check("[B-useful] for the day-before max temp the weight times the scoring-set "
      "standard deviation is 4.96, and so is sqrt(I/2)",
      (round(float(wstd[max_before]), 2), round(float(sqrthalf[max_before]), 2))
      == (4.96, 4.96))
with open(OUT_DIR / "featureimportance_useful.csv", "w") as f:
    f.write("label,corr,linear,forest,wstd,sqrthalf,alg1\n")
    for i in order[::-1]:
        f.write(f"{labels[i]},{corr[i]:.2f},{perm[i]:.2f},{perm_rf[i]:.2f},"
                f"{wstd[i]:.2f},{sqrthalf[i]:.2f},{score_std[i]:.2f}\n")

# ---- preview figure (checking only; the entry reads the CSVs)
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(7.0, 11.0))
d = np.arange(1, 31)
ya = np.array([y[i] for i in june])
yp = np.array([pred[i] for i in june])
ax1.vlines(d, 0, ya, color="0.6", linewidth=1)
ax1.plot(d, ya, "o", color="0.3", label="actual")
ax1.plot(d, yp, "x", color="black", markersize=7, label="predicted")
ax1.set_xlabel("day in June 2024")
ax1.set_ylabel("max daytime temp (deg C)")
ax1.set_title("Krems 2024: actual vs. predicted next-day max temperature")
ax1.legend(frameon=False)
pos = np.arange(12)
ax2.vlines(pos, 0, perm[order], color="0.6", linewidth=1)
ax2.plot(pos, perm[order], "x", color="black", markersize=7,
         label="permutation importance")
ax2.set_xticks(pos)
ax2.set_xticklabels([labels[i] for i in order], fontsize=8,
                    rotation=45, ha="right")
ax2.set_xlabel("feature")
ax2.set_ylabel("permutation importance (squared deg C)")
ax2.set_title("Permutation importance for the twelve features (linear)")
ax2.legend(frameon=False)
ax3.vlines(pos, 0, perm_rf[order], color="0.6", linewidth=1)
ax3.plot(pos, perm_rf[order], "o", color="black", markersize=5,
         label="permutation importance, random forest")
ax3.set_xticks(pos)
ax3.set_xticklabels([labels[i] for i in order], fontsize=8,
                    rotation=45, ha="right")
ax3.set_xlabel("feature")
ax3.set_ylabel("permutation importance (squared deg C)")
ax3.set_title("Permutation importance for the twelve features (random forest)")
ax3.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "featureimportance.png", dpi=110)

failed = [name for name, ok in report if not ok]
print(f"{len(report) - len(failed)}/{len(report)} checks passed"
      + (f"; FAILED: {failed}" if failed else ""))